In [1]:
# STANDIN A0
%run 00_setup.ipynb

Python 3.13.7, scikit-learn 1.6.1, numpy 2.5.3, pandas 3.0.6
Working directory: /home/steven/Desktop/AI-For-Cybersecurity-Lab-4.2-Robustness-attacks


train 267,984 rows   attack rate 0.1507
val    89,328 rows   attack rate 0.1507
test   89,329 rows   attack rate 0.1507


68 features: 58 continuous, 10 flag columns
Flag columns: Fwd PSH Flags, Fwd URG Flags, FIN Flag Count, SYN Flag Count, RST Flag Count, PSH Flag Count, ACK Flag Count, URG Flag Count, CWE Flag Count, ECE Flag Count
  tree    loaded from models/tree.joblib
  logreg  loaded from models/logreg.joblib


  forest  loaded from models/forest.joblib


ATTACK_IDX: 50 detected attacks, forest probability 1.000-1.000
  attack types: {'DDoS': 25, 'DoS Hulk': 22, 'DoS GoldenEye': 2, 'SSH-Patator': 1}
EXPLAIN_IDX: 500 random test flows, 68 of them attacks
Setup complete: 20 shared names ready. The models are scored in step A2.


<!-- STEP A1 -->
## Part A: The baseline

### Step A1: Check the data is healthy

Infinite or missing values would make the noise (Part B) and the attack (Part C) crash or produce
nonsense, so we check for them before anything else. We also count which columns are continuous and
which are 0/1 flags, because Part B adds noise to continuous columns only.

In [2]:
# STEP A1
print(f"X_train {X_train.shape}, X_val {X_val.shape}, X_test {X_test.shape}")
print(f"Share of attacks in the training labels: {y_train.mean():.4f}")

# np.isfinite is False for +inf, -inf and NaN, so one check covers infinities and missing values.
finite = {name: bool(np.isfinite(part).all().all())
          for name, part in (("train", X_train), ("val", X_val), ("test", X_test))}
print("All values finite:", finite)

if not all(finite.values()):
    # Not needed for our data (Lab 1 dropped every row with an infinity or NaN), but this is the fix:
    # turn infinities into NaN, then fill with the TRAINING median, never a test-set statistic.
    for part in (X_train, X_val, X_test):
        part.replace([np.inf, -np.inf], np.nan, inplace=True)
        part.fillna(TRAIN_MEDIAN, inplace=True)
assert all(np.isfinite(part).all().all() for part in (X_train, X_val, X_test))

print(f"\n{len(CONT_COLS)} continuous columns (more than two distinct values in training), "
      f"{len(FLAG_COLS)} flag columns:")
print("  " + ", ".join(FLAG_COLS))

X_train (267984, 68), X_val (89328, 68), X_test (89329, 68)
Share of attacks in the training labels: 0.1507


All values finite: {'train': True, 'val': True, 'test': True}



58 continuous columns (more than two distinct values in training), 10 flag columns:
  Fwd PSH Flags, Fwd URG Flags, FIN Flag Count, SYN Flag Count, RST Flag Count, PSH Flag Count, ACK Flag Count, URG Flag Count, CWE Flag Count, ECE Flag Count


In [3]:
# STEP A1
# Two things the later steps need to know about the continuous columns: are some values negative
# (matters for clipping noise at zero in B1), and how heavy-tailed are they (a few enormous values).
q = X_train[CONT_COLS].quantile([0.5, 0.99])
health = pd.DataFrame({
    "min": X_train[CONT_COLS].min(),
    "median": q.loc[0.5],
    "p99": q.loc[0.99],
    "max": X_train[CONT_COLS].max(),
    "max_over_p99": X_train[CONT_COLS].max() / q.loc[0.99].replace(0, np.nan),
})
negative = health.index[health["min"] < 0].tolist()
print(f"{len(negative)} continuous columns contain negative values in training:")
print("  " + ", ".join(negative))
print("\nMost heavy-tailed columns (maximum compared with the 99th percentile):")
print(health.sort_values("max_over_p99", ascending=False).head(5)[["median", "p99", "max"]]
      .to_string(float_format=lambda v: f"{v:,.0f}"))

summary = pd.DataFrame([{
    "train_rows": len(X_train), "val_rows": len(X_val), "test_rows": len(X_test),
    "features": len(FEATURES), "continuous": len(CONT_COLS), "flags": len(FLAG_COLS),
    "attack_rate_train": round(y_train.mean(), 4), "attack_rate_test": round(y_test.mean(), 4),
    "all_finite": all(finite.values()), "columns_with_negatives": len(negative),
}])
summary.to_csv("results/tables/A1_data_health.csv", index=False)
summary.T.rename(columns={0: "value"})

11 continuous columns contain negative values in training:
  Flow Duration, Flow Bytes/s, Flow Packets/s, Flow IAT Mean, Flow IAT Max, Flow IAT Min, Fwd Header Length, Bwd Header Length, Init_Win_bytes_forward, Init_Win_bytes_backward, min_seg_size_forward

Most heavy-tailed columns (maximum compared with the 99th percentile):
                             median    p99         max
Subflow Bwd Bytes               190 91,561 593,469,129
Total Length of Bwd Packets     190 91,561 593,000,000
act_data_pkt_fwd                  1     32     192,326
Subflow Bwd Packets               2     69     272,353
Total Backward Packets            2     69     272,353


,value
train_rows,267984
val_rows,89328
test_rows,89329
features,68
continuous,58
flags,10
attack_rate_train,0.1507
attack_rate_test,0.1507
all_finite,True
columns_with_negatives,11


<!-- STEP A1 -->
**What we found.** About 15% of the flows are attacks in every part of the split. Every value is
finite: Lab 1 already dropped the rows where `Flow Bytes/s` or `Flow Packets/s` were infinite (a
division by a zero duration), so nothing had to be fixed here. 58 of the 68 columns are continuous and
only the 10 flag columns are 0/1. That is the opposite of the phishing data in Lab 4.1, where many
features were flags.

Two warnings for Part B. Several continuous columns contain negative values in the raw data (−1 means
"not recorded", and the header lengths have a known CICFlowMeter bug), so clipping noisy values at
zero would also change real data. And the columns are heavy-tailed: the largest values are many times
the 99th percentile, so noise sized by the standard deviation will hit rows unevenly.

<!-- STEP A2 -->
### Step A2: Score the three models, and the model that does nothing

`report()` (from the setup) already includes PR-AUC, so it returns macro-F1, recall, ROC-AUC, PR-AUC
and FAR, plus accuracy for comparison. The "always benign" model is scikit-learn's
`DummyClassifier(strategy="constant", constant=0)`: it answers *benign* for every flow, so it goes
through the same `report()` as the real models. Its accuracy is `1 - y_test.mean()` and its recall
is zero.

In [4]:
# STEP A2
from sklearn.dummy import DummyClassifier

always_benign = DummyClassifier(strategy="constant", constant=0).fit(X_train, y_train)

baseline = pd.DataFrame(
    {name: report(model, X_test, y_test)
     for name, model in [("always benign", always_benign), *MODELS.items()]}).T
baseline.index.name = "model"

# The always-benign accuracy and recall must be exactly what the lab says.
assert np.isclose(baseline.loc["always benign", "accuracy"], 1 - y_test.mean())
assert baseline.loc["always benign", "recall"] == 0

baseline.to_csv("results/tables/A2_baseline.csv", float_format="%.4f")
baseline.round(4)

,accuracy,macro_f1,recall,roc_auc,pr_auc,far
model,,,,,,
always benign,0.8493,0.4593,0.0000,0.5000,0.1507,0.0000
tree,0.9985,0.9970,0.9950,0.9971,0.9907,0.0009
logreg,0.9720,0.9417,0.8326,0.9914,0.9693,0.0032
forest,0.9984,0.9968,0.9923,0.9999,0.9998,0.0006


In [5]:
# STEP A2
# Numbers for the two "Check yourself" questions.
acc_benign = baseline.loc["always benign", "accuracy"]
acc_forest = baseline.loc["forest", "accuracy"]
print(f"Accuracy: always benign {acc_benign:.4f}, forest {acc_forest:.4f}, "
      f"margin {acc_forest - acc_benign:+.4f}")
print(f"Errors:   always benign {1 - acc_benign:.4f}, forest {1 - acc_forest:.4f} "
      f"(the forest removes {1 - (1 - acc_forest) / (1 - acc_benign):.1%} of the errors)")
print(f"Recall:   always benign 0.0000, forest {baseline.loc['forest', 'recall']:.4f}")

real = baseline.drop(index="always benign")
print("\nBetween the three real models:")
for col in ("roc_auc", "pr_auc"):
    print(f"  {col:<8} from {real[col].min():.4f} to {real[col].max():.4f}  "
          f"(spread {real[col].max() - real[col].min():.4f})")
print(f"Gap between ROC-AUC and PR-AUC per model:")
print((real["roc_auc"] - real["pr_auc"]).round(4).to_string())

Accuracy: always benign 0.8493, forest 0.9984, margin +0.1491
Errors:   always benign 0.1507, forest 0.0016 (the forest removes 98.9% of the errors)
Recall:   always benign 0.0000, forest 0.9923

Between the three real models:
  roc_auc  from 0.9914 to 0.9999  (spread 0.0085)
  pr_auc   from 0.9693 to 0.9998  (spread 0.0305)
Gap between ROC-AUC and PR-AUC per model:
model
tree      0.0064
logreg    0.0222
forest    0.0002


<!-- STEP A2 -->
**What we found.** The always-benign model is **84.9% accurate** and catches **no attack at all**
(recall 0, macro-F1 0.46, PR-AUC 0.15, which equals the attack rate). Every later number in this lab
is compared with this table.

**Check yourself 1: how far above the always-benign accuracy is the forest?** 0.9984 against 0.8493,
a margin of +0.149. That sounds like "15 points better", but it hides most of the story. The honest
comparisons are the errors and the recall: the forest makes 0.16% errors instead of 15.07% (it removes
about 99% of them), and it catches 99.2% of the attacks instead of none. With attacks at 15% the
imbalance is milder than the lab's 2% example, so accuracy is less misleading here, but it still
makes a model that does nothing look good.

**Check yourself 2: ROC-AUC or PR-AUC, which moved more?** PR-AUC. Across the three models ROC-AUC only
spans 0.991–1.000 (a spread of 0.009), while PR-AUC spans 0.969–1.000 (0.031, more than three times as
much). Logistic regression loses 0.022 between its ROC-AUC and its PR-AUC, the forest only 0.0002.
ROC-AUC is built from the false-positive *rate*, which is divided by the 75,868 benign test flows, so
a few hundred false alarms barely move it. PR-AUC uses *precision*, which compares those false
alarms with the 13,461 attacks, so the same false alarms cost much more. That is why the lab says to
report PR-AUC on imbalanced data.

**Side note.** The single tree has the best macro-F1 (0.9970, against 0.9968 for the forest), but the
lowest ROC-AUC of the three real models. A fully grown tree gives almost every flow a probability of
exactly 0 or 1, so it decides well but cannot rank flows by how suspicious they are. Part B will show
whether its decisions also survive messy data.